In [2]:
from flask import Flask, render_template, request, jsonify, send_file
from werkzeug.utils import secure_filename
from groq import Groq
from twilio.rest import Client
from linebot import LineBotApi
from linebot.models import TextSendMessage
from google.oauth2 import service_account
from googleapiclient.discovery import build
import base64
import io
from PIL import Image
import csv
from datetime import datetime
import pytz
import os
import threading
import uuid
import cv2

app = Flask(__name__)
app.config['UPLOAD_FOLDER'] = 'uploads'
app.config['MAX_CONTENT_LENGTH'] = 500 * 1024 * 1024  # 500MB max file size for videos
app.config['ALLOWED_EXTENSIONS'] = {'png', 'jpg', 'jpeg', 'gif', 'bmp', 'mp4', 'avi', 'mov', 'mkv', 'webm'}

# 確保上傳資料夾存在
os.makedirs(app.config['UPLOAD_FOLDER'], exist_ok=True)

# Google Sheets 設定
SERVICE_ACCOUNT_FILE = 'key.json'
SCOPES = ['https://www.googleapis.com/auth/spreadsheets']
SPREADSHEET_ID = '1rZ_557DFPdlmVaDEn4zYubb-ejntHunPfHG1QiXEjDo'

# 全域變數儲存辨識結果
recognition_results = []
processing_status = {
    'is_processing': False,
    'current': 0,
    'total': 0,
    'current_file': ''
}

# 台灣時區
TZ = pytz.timezone('Asia/Taipei')

# API 限制常數
MAX_IMAGES_PER_REQUEST = 5

def get_taiwan_time():
    """取得台灣時間"""
    return datetime.now(TZ).strftime("%Y-%m-%d %H:%M:%S")

def allowed_file(filename):
    """檢查檔案類型是否允許"""
    return '.' in filename and filename.rsplit('.', 1)[1].lower() in app.config['ALLOWED_EXTENSIONS']

def is_video_file(filename):
    """檢查是否為影片檔案"""
    video_extensions = {'mp4', 'avi', 'mov', 'mkv', 'webm'}
    return '.' in filename and filename.rsplit('.', 1)[1].lower() in video_extensions

def encode_image(image):
    """將圖片編碼為 base64"""
    buffered = io.BytesIO()
    image.save(buffered, format="JPEG")
    return base64.b64encode(buffered.getvalue()).decode("utf-8")

def extract_frames_from_video(video_path, interval_seconds=3):
    """從影片中提取幀
    
    Args:
        video_path: 影片檔案路徑
        interval_seconds: 提取幀的間隔秒數
    
    Returns:
        提取的幀列表（PIL Image 格式）
    """
    frames = []
    try:
        cap = cv2.VideoCapture(video_path)
        fps = cap.get(cv2.CAP_PROP_FPS)
        
        if fps == 0:
            print(f"無法取得影片幀率: {video_path}")
            cap.release()
            return frames
        
        frame_interval = int(fps * interval_seconds)
        frame_count = 0
        extracted_count = 0
        
        while True:
            ret, frame = cap.read()
            if not ret:
                break
            
            if frame_count % frame_interval == 0:
                # 將 BGR 轉換為 RGB
                frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                # 轉換為 PIL Image
                pil_image = Image.fromarray(frame_rgb)
                frames.append(pil_image)
                extracted_count += 1
            
            frame_count += 1
        
        cap.release()
        print(f"從影片提取了 {extracted_count} 個幀")
        
    except Exception as e:
        print(f"影片處理錯誤: {str(e)}")
    
    return frames

def get_google_sheet_service():
    """建立 Google Sheets 服務連線"""
    try:
        creds = service_account.Credentials.from_service_account_file(
            SERVICE_ACCOUNT_FILE, scopes=SCOPES)
        service = build('sheets', 'v4', credentials=creds)
        return service
    except Exception as e:
        print(f"Google Sheets 連線失敗: {str(e)}")
        return None

def get_first_sheet_title(service, spreadsheet_id):
    """從 Google 試算表中獲取第一個分頁的名稱"""
    try:
        sheet_metadata = service.spreadsheets().get(spreadsheetId=spreadsheet_id).execute()
        sheets = sheet_metadata.get('sheets', [])
        if not sheets:
            raise ValueError("試算表中沒有任何分頁！")
        first_sheet_title = sheets[0].get("properties", {}).get("title", "Sheet1")
        print(f"成功取得第一個分頁名稱：'{first_sheet_title}'")
        return first_sheet_title
    except Exception as e:
        print(f"無法取得第一個分頁名稱: {str(e)}")
        return "Sheet1"

def write_to_google_sheet(timestamp, filename, result):
    """將辨識結果寫入 Google Sheets"""
    try:
        service = get_google_sheet_service()
        if not service:
            return False
        
        sheet_title = get_first_sheet_title(service, SPREADSHEET_ID)
        
        # 準備要寫入的資料
        values = [[timestamp, filename, result]]
        
        # 附加到現有資料後面
        body = {'values': values}
        service.spreadsheets().values().append(
            spreadsheetId=SPREADSHEET_ID,
            range=f"{sheet_title}!A:C",
            valueInputOption="USER_ENTERED",
            body=body
        ).execute()
        
        print(f"成功寫入 Google Sheets: {filename}")
        return True
    except Exception as e:
        print(f"Google Sheets 寫入失敗: {str(e)}")
        return False

@app.route('/')
def index():
    """首頁"""
    return render_template('index.html')

@app.route('/api/upload', methods=['POST'])
def upload_files():
    """上傳圖片或影片"""
    if 'files[]' not in request.files:
        return jsonify({'success': False, 'error': '沒有選擇檔案'}), 400
    
    files = request.files.getlist('files[]')
    interval_seconds = int(request.form.get('interval', 3))  # 預設 3 秒
    uploaded_files = []
    
    for file in files:
        if file and allowed_file(file.filename):
            filename = secure_filename(file.filename)
            unique_filename = f"{uuid.uuid4().hex}_{filename}"
            filepath = os.path.join(app.config['UPLOAD_FOLDER'], unique_filename)
            file.save(filepath)
            
            # 檢查是否為影片檔案
            if is_video_file(filename):
                # 提取影片幀
                frames = extract_frames_from_video(filepath, interval_seconds)
                
                # 將每個幀儲存為獨立圖片
                for idx, frame in enumerate(frames):
                    frame_filename = f"{uuid.uuid4().hex}_frame_{idx:04d}.jpg"
                    frame_path = os.path.join(app.config['UPLOAD_FOLDER'], frame_filename)
                    frame.save(frame_path, 'JPEG', quality=85)
                    
                    uploaded_files.append({
                        'original_name': f"{filename}_frame_{idx:04d}",
                        'stored_name': frame_filename,
                        'path': frame_path,
                        'type': 'video_frame'
                    })
                
                # 刪除原始影片檔案以節省空間
                try:
                    os.remove(filepath)
                except:
                    pass
            else:
                # 一般圖片
                uploaded_files.append({
                    'original_name': filename,
                    'stored_name': unique_filename,
                    'path': filepath,
                    'type': 'image'
                })
    
    return jsonify({
        'success': True,
        'files': uploaded_files,
        'count': len(uploaded_files)
    })

@app.route('/api/analyze', methods=['POST'])
def analyze_images():
    """批次辨識圖片"""
    global recognition_results, processing_status
    
    data = request.get_json()
    
    api_key = data.get('api_key', '').strip()
    prompt = data.get('prompt', '').strip()
    files = data.get('files', [])
    analyze_mode = data.get('analyze_mode', 'individual')  # 'individual'、'sequence' 或 'grouped_sequence'
    recording_groups = data.get('recording_groups', None)
    
    # Twilio 設定
    twilio_config = {
        'account_sid': data.get('twilio_sid', '').strip(),
        'auth_token': data.get('twilio_token', '').strip(),
        'from_number': data.get('twilio_from', '').strip(),
        'to_number': data.get('twilio_to', '').strip(),
        'enabled': data.get('sms_enabled', False)
    }
    
    # LINE Bot 設定
    line_config = {
        'channel_access_token': data.get('line_token', '').strip(),
        'user_id': data.get('line_user_id', '').strip(),
        'enabled': data.get('line_enabled', False)
    }
    
    # Google Sheets 設定
    google_sheets_enabled = data.get('google_sheets_enabled', False)
    
    if not api_key:
        return jsonify({'success': False, 'error': '請輸入 API 金鑰'}), 400
    
    if not prompt:
        return jsonify({'success': False, 'error': '請輸入提示詞'}), 400
    
    if not files:
        return jsonify({'success': False, 'error': '沒有圖片可辨識'}), 400
    
    # 重置狀態
    recognition_results = []
    
    # 根據模式選擇處理方式
    if analyze_mode == 'grouped_sequence' and recording_groups:
        # 分組序列分析模式
        processing_status = {
            'is_processing': True,
            'current': 0,
            'total': len(recording_groups),
            'current_file': ''
        }
        thread = threading.Thread(
            target=analyze_grouped_sequences,
            args=(api_key, prompt, recording_groups, twilio_config, line_config, google_sheets_enabled)
        )
    elif analyze_mode == 'sequence':
        # 單一序列分析模式
        processing_status = {
            'is_processing': True,
            'current': 0,
            'total': len(files),
            'current_file': ''
        }
        thread = threading.Thread(
            target=analyze_image_sequence,
            args=(api_key, prompt, files, twilio_config, line_config, google_sheets_enabled)
        )
    else:
        # 個別圖片分析模式
        processing_status = {
            'is_processing': True,
            'current': 0,
            'total': len(files),
            'current_file': ''
        }
        thread = threading.Thread(
            target=batch_process_images,
            args=(api_key, prompt, files, twilio_config, line_config, google_sheets_enabled)
        )
    
    thread.start()
    return jsonify({'success': True, 'message': '開始批次辨識'})

def analyze_grouped_sequences(api_key, prompt, groups, twilio_config, line_config, google_sheets_enabled):
    """分析分組的圖片序列（攝像頭錄影分組）"""
    global recognition_results, processing_status
    
    client = Groq(api_key=api_key)
    
    for group_index, group_info in enumerate(groups):
        try:
            processing_status['current'] = group_index + 1
            group_id = group_info['groupId']
            group_files = group_info['files']
            
            # API 限制：最多 5 張圖片
            original_count = len(group_files)
            if original_count > MAX_IMAGES_PER_REQUEST:
                print(f"警告：第 {group_index + 1} 組有 {original_count} 張圖片，超過 API 限制（{MAX_IMAGES_PER_REQUEST}張），將只使用前 {MAX_IMAGES_PER_REQUEST} 張")
                group_files = group_files[:MAX_IMAGES_PER_REQUEST]
            
            processing_status['current_file'] = f'分析第 {group_index + 1} 組 ({len(group_files)} 張圖片)'
            
            # 讀取該組的所有圖片
            image_contents = []
            missing_files = []
            for file_info in group_files:
                image_path = file_info['path']
                
                # 檢查文件是否存在
                if not os.path.exists(image_path):
                    missing_files.append(file_info.get('original_name', image_path))
                    print(f"警告：文件不存在: {image_path}")
                    continue
                
                try:
                    image = Image.open(image_path)
                    base64_image = encode_image(image)
                    image_contents.append({
                        "type": "image_url",
                        "image_url": {"url": f"data:image/jpeg;base64,{base64_image}"}
                    })
                except Exception as e:
                    print(f"讀取圖片失敗 {image_path}: {str(e)}")
                    missing_files.append(file_info.get('original_name', image_path))
            
            # 如果沒有有效的圖片，跳過此組
            if not image_contents:
                error_msg = f"第 {group_index + 1} 組沒有有效的圖片文件"
                if missing_files:
                    error_msg += f"（缺少 {len(missing_files)} 個文件，可能已被清除）"
                timestamp = get_taiwan_time()
                recognition_results.append({
                    'timestamp': timestamp,
                    'filename': f"錄影片段 {group_index + 1}",
                    'result': error_msg
                })
                continue
            
            # 構建提示詞
            sequence_prompt = f"""{prompt}

這是第 {group_index + 1} 組的 {len(image_contents)} 張連續圖片。請分析這組圖片，並描述：
1. 這段時間內發生了什麼事情
2. 主要的動作和變化
3. 場景中的重要物體和人物
4. 事件的順序和進展

請提供一個清晰、完整的描述。"""
            
            # 構建消息內容
            message_content = [{"type": "text", "text": sequence_prompt}]
            message_content.extend(image_contents)
            
            # 調用 API
            completion = client.chat.completions.create(
                model="meta-llama/llama-4-scout-17b-16e-instruct",
                messages=[{
                    "role": "user",
                    "content": message_content
                }],
                temperature=0.7,
                max_completion_tokens=4096,
                top_p=1,
                stream=False,
                stop=None,
            )
            
            result = completion.choices[0].message.content
            
            # 儲存結果
            timestamp = get_taiwan_time()
            group_name = f"錄影片段 {group_index + 1} ({len(image_contents)} 張圖片)"
            if original_count > MAX_IMAGES_PER_REQUEST:
                group_name += f" [原{original_count}張，已限制為{MAX_IMAGES_PER_REQUEST}張]"
            if missing_files:
                group_name += f" [跳過{len(missing_files)}個無效文件]"
            
            recognition_results.append({
                'timestamp': timestamp,
                'filename': group_name,
                'result': result
            })
            
            # 寫入 Google Sheets
            if google_sheets_enabled:
                write_to_google_sheet(timestamp, group_name, result)
            
            # 發送簡訊
            if twilio_config['enabled'] and all([
                twilio_config['account_sid'],
                twilio_config['auth_token'],
                twilio_config['from_number'],
                twilio_config['to_number']
            ]):
                send_sms(twilio_config, result, group_name, timestamp)
            
            # 發送 LINE 訊息
            if line_config['enabled'] and all([
                line_config['channel_access_token'],
                line_config['user_id']
            ]):
                send_line_message(line_config, result, group_name, timestamp)
            
        except Exception as e:
            error_msg = f"第 {group_index + 1} 組分析錯誤: {str(e)}"
            timestamp = get_taiwan_time()
            recognition_results.append({
                'timestamp': timestamp,
                'filename': f"錄影片段 {group_index + 1}",
                'result': error_msg
            })
            
            if google_sheets_enabled:
                write_to_google_sheet(timestamp, f"錄影片段 {group_index + 1}", error_msg)
    
    processing_status['is_processing'] = False

def analyze_image_sequence(api_key, prompt, files, twilio_config, line_config, google_sheets_enabled):
    """分析圖片序列（影片或錄影片段）並生成連貫描述"""
    global recognition_results, processing_status
    
    client = Groq(api_key=api_key)
    
    try:
        processing_status['current'] = 1
        
        # API 限制：最多 5 張圖片
        original_count = len(files)
        if original_count > MAX_IMAGES_PER_REQUEST:
            print(f"警告：序列有 {original_count} 張圖片，超過 API 限制（{MAX_IMAGES_PER_REQUEST}張），將只使用前 {MAX_IMAGES_PER_REQUEST} 張")
            files = files[:MAX_IMAGES_PER_REQUEST]
        
        processing_status['current_file'] = f'分析 {len(files)} 張圖片序列'
        
        # 讀取所有圖片
        image_contents = []
        missing_files = []
        for file_info in files:
            image_path = file_info['path']
            
            # 檢查文件是否存在
            if not os.path.exists(image_path):
                missing_files.append(file_info.get('original_name', image_path))
                print(f"警告：文件不存在: {image_path}")
                continue
            
            try:
                image = Image.open(image_path)
                base64_image = encode_image(image)
                image_contents.append({
                    "type": "image_url",
                    "image_url": {"url": f"data:image/jpeg;base64,{base64_image}"}
                })
            except Exception as e:
                print(f"讀取圖片失敗 {image_path}: {str(e)}")
                missing_files.append(file_info.get('original_name', image_path))
        
        # 如果沒有有效的圖片，返回錯誤
        if not image_contents:
            error_msg = "序列中沒有有效的圖片文件"
            if missing_files:
                error_msg += f"（缺少 {len(missing_files)} 個文件，可能已被清除）"
            timestamp = get_taiwan_time()
            recognition_results.append({
                'timestamp': timestamp,
                'filename': '影片序列分析',
                'result': error_msg
            })
            processing_status['is_processing'] = False
            return
        
        # 構建提示詞
        sequence_prompt = f"""{prompt}

這是一段時間序列的 {len(image_contents)} 張連續圖片。請分析這些圖片，並描述：
1. 整段時間內發生了什麼事情
2. 主要的動作和變化
3. 場景中的重要物體和人物
4. 時間順序和事件進展

請提供一個連貫、完整的描述，就像在講述一個故事一樣。"""
        
        # 構建消息內容
        message_content = [{"type": "text", "text": sequence_prompt}]
        message_content.extend(image_contents)
        
        # 調用 API
        completion = client.chat.completions.create(
            model="meta-llama/llama-4-scout-17b-16e-instruct",
            messages=[{
                "role": "user",
                "content": message_content
            }],
            temperature=0.7,
            max_completion_tokens=4096,
            top_p=1,
            stream=False,
            stop=None,
        )
        
        result = completion.choices[0].message.content
        
        # 儲存結果
        timestamp = get_taiwan_time()
        sequence_name = f"{files[0]['original_name'].split('_')[0]}_sequence ({len(image_contents)} frames)"
        if original_count > MAX_IMAGES_PER_REQUEST:
            sequence_name += f" [原{original_count}張，已限制為{MAX_IMAGES_PER_REQUEST}張]"
        if missing_files:
            sequence_name += f" [跳過{len(missing_files)}個無效文件]"
        
        recognition_results.append({
            'timestamp': timestamp,
            'filename': sequence_name,
            'result': result
        })
        
        # 寫入 Google Sheets
        if google_sheets_enabled:
            write_to_google_sheet(timestamp, sequence_name, result)
        
        # 發送簡訊
        if twilio_config['enabled'] and all([
            twilio_config['account_sid'],
            twilio_config['auth_token'],
            twilio_config['from_number'],
            twilio_config['to_number']
        ]):
            send_sms(twilio_config, result, sequence_name, timestamp)
        
        # 發送 LINE 訊息
        if line_config['enabled'] and all([
            line_config['channel_access_token'],
            line_config['user_id']
        ]):
            send_line_message(line_config, result, sequence_name, timestamp)
        
    except Exception as e:
        error_msg = f"序列分析錯誤: {str(e)}"
        timestamp = get_taiwan_time()
        recognition_results.append({
            'timestamp': timestamp,
            'filename': '影片序列分析',
            'result': error_msg
        })
        
        if google_sheets_enabled:
            write_to_google_sheet(timestamp, '影片序列分析', error_msg)
    
    processing_status['is_processing'] = False

def batch_process_images(api_key, prompt, files, twilio_config, line_config, google_sheets_enabled):
    """批次處理圖片（逐張分析）"""
    global recognition_results, processing_status
    
    client = Groq(api_key=api_key)
    
    for index, file_info in enumerate(files):
        try:
            processing_status['current'] = index + 1
            processing_status['current_file'] = file_info['original_name']
            
            # 檢查文件是否存在
            image_path = file_info['path']
            if not os.path.exists(image_path):
                error_msg = f"錯誤: 文件不存在（可能已被清除）"
                timestamp = get_taiwan_time()
                recognition_results.append({
                    'timestamp': timestamp,
                    'filename': file_info['original_name'],
                    'result': error_msg
                })
                
                if google_sheets_enabled:
                    write_to_google_sheet(timestamp, file_info['original_name'], error_msg)
                continue
            
            # 讀取並辨識圖片
            image = Image.open(image_path)
            base64_image = encode_image(image)
            
            image_content = {
                "type": "image_url",
                "image_url": {"url": f"data:image/jpeg;base64,{base64_image}"}
            }
            
            # 增強提示詞
            enhanced_prompt = f"{prompt}\n\n請提供詳細且完整的描述，包含所有重要細節、顏色、位置、數量等資訊。"
            
            completion = client.chat.completions.create(
                model="meta-llama/llama-4-scout-17b-16e-instruct",
                messages=[{
                    "role": "user",
                    "content": [
                        {"type": "text", "text": enhanced_prompt},
                        image_content
                    ]
                }],
                temperature=0.7,
                max_completion_tokens=2048,
                top_p=1,
                stream=False,
                stop=None,
            )
            
            result = completion.choices[0].message.content
            
            # 儲存結果
            timestamp = get_taiwan_time()
            recognition_results.append({
                'timestamp': timestamp,
                'filename': file_info['original_name'],
                'result': result
            })
            
            # 寫入 Google Sheets
            if google_sheets_enabled:
                write_to_google_sheet(timestamp, file_info['original_name'], result)
            
            # 發送簡訊
            if twilio_config['enabled'] and all([
                twilio_config['account_sid'],
                twilio_config['auth_token'],
                twilio_config['from_number'],
                twilio_config['to_number']
            ]):
                send_sms(twilio_config, result, file_info['original_name'], timestamp)
            
            # 發送 LINE 訊息
            if line_config['enabled'] and all([
                line_config['channel_access_token'],
                line_config['user_id']
            ]):
                send_line_message(line_config, result, file_info['original_name'], timestamp)
            
        except Exception as e:
            error_msg = f"錯誤: {str(e)}"
            timestamp = get_taiwan_time()
            recognition_results.append({
                'timestamp': timestamp,
                'filename': file_info['original_name'],
                'result': error_msg
            })
            
            if google_sheets_enabled:
                write_to_google_sheet(timestamp, file_info['original_name'], error_msg)
    
    processing_status['is_processing'] = False

def send_sms(config, result, image_name, timestamp):
    """發送簡訊"""
    try:
        client = Client(config['account_sid'], config['auth_token'])
        
        message_body = f"【圖片辨識通知】\n"
        message_body += f"圖片: {image_name}\n"
        message_body += f"時間: {timestamp}\n\n"
        message_body += f"結果: {result[:100]}..."
        
        message = client.messages.create(
            from_=config['from_number'],
            to=config['to_number'],
            body=message_body
        )
        
        print(f"簡訊已發送 - SID: {message.sid}")
        
    except Exception as e:
        print(f"簡訊發送失敗: {str(e)}")

def send_line_message(config, result, image_name, timestamp):
    """發送 LINE 訊息"""
    try:
        line_bot = LineBotApi(config['channel_access_token'])
        
        message_body = f"【圖片辨識通知】\n"
        message_body += f"圖片: {image_name}\n"
        message_body += f"時間: {timestamp}\n\n"
        message_body += f"結果:\n{result[:500]}"
        
        if len(result) > 500:
            message_body += "..."
        
        line_bot.push_message(
            config['user_id'],
            TextSendMessage(text=message_body)
        )
        
        print(f"LINE 訊息已發送給: {config['user_id']}")
        
    except Exception as e:
        print(f"LINE 訊息發送失敗: {str(e)}")

@app.route('/api/status', methods=['GET'])
def get_status():
    """取得處理狀態"""
    return jsonify({
        'success': True,
        'status': processing_status,
        'results_count': len(recognition_results)
    })

@app.route('/api/results', methods=['GET'])
def get_results():
    """取得辨識結果"""
    return jsonify({
        'success': True,
        'results': recognition_results
    })

@app.route('/api/export-csv', methods=['GET'])
def export_csv():
    """匯出 CSV"""
    if not recognition_results:
        return jsonify({'success': False, 'error': '沒有辨識結果可匯出'}), 400
    
    output = io.StringIO()
    writer = csv.writer(output)
    writer.writerow(['時間戳記 (台灣時間)', '圖片檔名', '辨識結果'])
    
    for result in recognition_results:
        writer.writerow([
            result['timestamp'],
            result['filename'],
            result['result']
        ])
    
    output.seek(0)
    csv_data = output.getvalue().encode('utf-8-sig')
    
    filename = f"recognition_results_{datetime.now(TZ).strftime('%Y%m%d_%H%M%S')}.csv"
    
    return send_file(
        io.BytesIO(csv_data),
        mimetype='text/csv',
        as_attachment=True,
        download_name=filename
    )

@app.route('/api/clear', methods=['POST'])
def clear_data():
    """清除所有資料"""
    global recognition_results, processing_status
    
    recognition_results = []
    processing_status = {
        'is_processing': False,
        'current': 0,
        'total': 0,
        'current_file': ''
    }
    
    # 清除上傳的檔案
    for filename in os.listdir(app.config['UPLOAD_FOLDER']):
        file_path = os.path.join(app.config['UPLOAD_FOLDER'], filename)
        try:
            if os.path.isfile(file_path):
                os.unlink(file_path)
        except Exception as e:
            print(f"刪除檔案失敗: {e}")
    
    return jsonify({'success': True, 'message': '已清除所有資料'})

if __name__ == '__main__':
    app.run(debug=True, host='0.0.0.0', port=5000, use_reloader=False)


 * Serving Flask app '__main__'
 * Debug mode: on


 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://192.168.68.105:5000
Press CTRL+C to quit
127.0.0.1 - - [08/Feb/2026 21:15:42] "GET / HTTP/1.1" 200 -


從影片提取了 14 個幀


127.0.0.1 - - [08/Feb/2026 21:15:57] "POST /api/upload HTTP/1.1" 200 -
127.0.0.1 - - [08/Feb/2026 21:16:14] "POST /api/analyze HTTP/1.1" 200 -
127.0.0.1 - - [08/Feb/2026 21:16:15] "GET /api/status HTTP/1.1" 200 -
127.0.0.1 - - [08/Feb/2026 21:16:16] "GET /api/status HTTP/1.1" 200 -
127.0.0.1 - - [08/Feb/2026 21:16:17] "GET /api/status HTTP/1.1" 200 -
127.0.0.1 - - [08/Feb/2026 21:16:18] "GET /api/status HTTP/1.1" 200 -
127.0.0.1 - - [08/Feb/2026 21:16:19] "GET /api/status HTTP/1.1" 200 -
127.0.0.1 - - [08/Feb/2026 21:16:20] "GET /api/status HTTP/1.1" 200 -
127.0.0.1 - - [08/Feb/2026 21:16:21] "GET /api/status HTTP/1.1" 200 -
127.0.0.1 - - [08/Feb/2026 21:16:22] "GET /api/status HTTP/1.1" 200 -
127.0.0.1 - - [08/Feb/2026 21:16:23] "GET /api/status HTTP/1.1" 200 -
127.0.0.1 - - [08/Feb/2026 21:16:24] "GET /api/status HTTP/1.1" 200 -
127.0.0.1 - - [08/Feb/2026 21:16:25] "GET /api/status HTTP/1.1" 200 -
127.0.0.1 - - [08/Feb/2026 21:16:26] "GET /api/status HTTP/1.1" 200 -
127.0.0.1 - - [08